# UNet — Deblur & Denoise (End-to-End)
Metodo deep learning con architettura encoder-decoder e skip connections per il restauro di immagini degradate.

**Architettura:** Encoder (64→512 canali) + Bottleneck (1024) + Decoder con skip connections.

**Training:** MSE loss, Adam ($lr=1\times 10^{-4}$), 50 epoche, multi-noise augmentation.

**Implementazione:** `src/methods/unet/unet.py`

In [ ]:
import sys
sys.path.insert(0, '..')
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm import tqdm
import pandas as pd
import time

from src.data.dataset import load_config, LBCDataset
from src.degradation.degradation import degrade
from src.methods.unet.unet import UNet
from src.eval.metrics import evaluate

config = load_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(config['seed'])
print(f'Device: {device}')
print('Config caricato correttamente.')

## Configurazione

In [ ]:
noise_levels = config['degradation']['noise_levels']
kernel_size  = config['degradation']['kernel_size']
blur_sigma   = config['degradation']['blur_sigma']
batch_size   = config['unet']['batch_size']
epochs       = config['unet']['epochs']
lr           = config['unet']['lr']

print(f'Noise levels : {noise_levels}')
print(f'Kernel       : size={kernel_size}, sigma={blur_sigma}')
print(f'Batch size   : {batch_size}')
print(f'Epochs       : {epochs}')
print(f'Learning rate: {lr}')

## Caricamento dataset

In [ ]:
train_dataset = LBCDataset('data/splits/train.txt', config['dataset']['image_size'])
val_dataset   = LBCDataset('data/splits/val.txt',   config['dataset']['image_size'])
test_dataset  = LBCDataset('data/splits/test.txt',  config['dataset']['image_size'])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)

print(f'Train : {len(train_dataset)} immagini')
print(f'Val   : {len(val_dataset)} immagini')
print(f'Test  : {len(test_dataset)} immagini')

## Modello
Inizializzazione UNet e conteggio parametri.

In [ ]:
model = UNet().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parametri totali: {n_params:,} (~{n_params/1e6:.1f}M)')

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.MSELoss()

## Training con multi-noise augmentation
Per ogni batch, viene campionato un noise level random tra i 4 disponibili. Questo rende il modello robusto a qualsiasi intensita' di rumore.

Dopo ogni epoca: validazione su val set, salvataggio del modello migliore.

In [ ]:
best_val_psnr = -float('inf')
best_epoch = 0
train_losses = []
val_psnrs = []

for epoch in range(epochs):
    # Training
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]', leave=False)
    for gt in pbar:
        gt = gt.to(device)
        # Multi-noise: campiona noise level random
        noise = np.random.choice(noise_levels)
        degraded = torch.stack([
            degrade(g.cpu(), kernel_size=kernel_size, sigma=blur_sigma, noise_level=noise)
            for g in gt
        ]).to(device)

        pred = model(degraded)
        loss = criterion(pred, gt)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    # Validation
    model.eval()
    val_psnr_total = 0.0
    with torch.no_grad():
        for val_noise in noise_levels:
            noise_psnr = 0.0
            for gt in val_loader:
                gt = gt.to(device)
                degraded = torch.stack([
                    degrade(g.cpu(), kernel_size=kernel_size, sigma=blur_sigma, noise_level=val_noise)
                    for g in gt
                ]).to(device)
                pred = model(degraded)
                for j in range(gt.size(0)):
                    m = evaluate(pred[j].cpu(), gt[j].cpu())
                    noise_psnr += m['psnr']
            val_psnr_total += noise_psnr / len(val_dataset)

    avg_val_psnr = val_psnr_total / len(noise_levels)
    val_psnrs.append(avg_val_psnr)

    improved = avg_val_psnr > best_val_psnr
    marker = ' << BEST' if improved else ''
    print(f'Epoch {epoch+1:2d}/{epochs} | Loss: {avg_loss:.4f} | Val PSNR: {avg_val_psnr:.2f} dB{marker}')

    if improved:
        best_val_psnr = avg_val_psnr
        best_epoch = epoch + 1
        torch.save(model.state_dict(), 'best_model_unet.pth')

print(f'\nTraining completato. Miglior modello: epoch {best_epoch} (PSNR: {best_val_psnr:.2f} dB)')

## Curve di training

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, epochs+1), train_losses, marker='o', color='#2196F3')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, epochs+1), val_psnrs, marker='o', color='#4CAF50')
ax2.axvline(x=best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('PSNR (dB)')
ax2.set_title('Validation PSNR')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Demo visivo
Confronto tra immagine degradata, ricostruzione UNet e ground truth su alcune immagini di esempio.

In [ ]:
# Carica il miglior modello salvato
model.load_state_dict(torch.load('best_model_unet.pth', weights_only=True))
model.eval()

n_examples  = 3
noise_level = 0.05

def to_display(t):
    """Tensore [-1,1] → numpy [0,1] per matplotlib."""
    img = t.detach().cpu().permute(1, 2, 0).numpy()
    return np.clip(img * 0.5 + 0.5, 0, 1)

fig, axes = plt.subplots(n_examples, 3, figsize=(12, 4 * n_examples))

for i in range(n_examples):
    gt       = test_dataset[i]
    degraded = degrade(gt, kernel_size=kernel_size, sigma=blur_sigma, noise_level=noise_level)

    print(f'Processando immagine {i+1}/{n_examples}...')
    with torch.no_grad():
        restored = model(degraded.unsqueeze(0).to(device)).squeeze(0).cpu()

    axes[i, 0].imshow(to_display(gt))
    axes[i, 0].set_title('Ground Truth')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(to_display(degraded))
    axes[i, 1].set_title(f'Degraded (noise={noise_level})')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(to_display(restored))
    axes[i, 2].set_title('UNet Restored')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## Valutazione quantitativa
Calcolo PSNR e SSIM su tutto il test set per tutti i livelli di rumore.

In [ ]:
results = []

for noise_level in noise_levels:
    psnr_list, ssim_list, time_list = [], [], []

    for i in tqdm(range(len(test_dataset)), desc=f'UNet noise={noise_level}'):
        gt       = test_dataset[i]
        degraded = degrade(gt, kernel_size=kernel_size, sigma=blur_sigma, noise_level=noise_level)

        t0 = time.time()
        with torch.no_grad():
            restored = model(degraded.unsqueeze(0).to(device)).squeeze(0).cpu()
        time_list.append(time.time() - t0)

        m = evaluate(restored, gt)
        psnr_list.append(m['psnr'])
        ssim_list.append(m['ssim'])

    results.append({
        'method'             : 'unet',
        'noise_level'        : noise_level,
        'psnr'               : np.mean(psnr_list),
        'ssim'               : np.mean(ssim_list),
        'avg_inference_time' : np.mean(time_list),
    })
    print(f"[UNet] noise={noise_level} | PSNR={results[-1]['psnr']:.2f} "
          f"| SSIM={results[-1]['ssim']:.4f} | Time={results[-1]['avg_inference_time']:.3f}s")

df_results = pd.DataFrame(results)
df_results

## Salvataggio risultati

In [ ]:
output_dir = Path(config['eval']['results_dir']) / 'unet'
output_dir.mkdir(parents=True, exist_ok=True)

df_results.to_csv(output_dir / 'metrics.csv', index=False)
print(f"Risultati salvati in {output_dir / 'metrics.csv'}")

## Confronto con altri metodi
Caricamento risultati TV e DiffPIR per confronto finale.

*Tabella popolata dopo esecuzione di tutti gli script.*

In [ ]:
from src.plots.visualize import plot_metrics

results_dir = Path(config['eval']['results_dir'])

def load_metrics(method):
    p = results_dir / method / 'metrics.csv'
    return pd.read_csv(p) if p.exists() else None

df_tv      = load_metrics('tv')
df_unet    = load_metrics('unet')
df_diffpir = load_metrics('diffpir')

dfs = [df for df in [df_tv, df_unet, df_diffpir] if df is not None]
if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    display(df_all.pivot(index='noise_level', columns='method', values=['psnr', 'ssim']))
    plot_metrics(results_dir, save_path=results_dir / 'comparison.png')
else:
    print('Esegui prima TV e DiffPIR per il confronto completo.')

## Discussione

### Risultati attesi
- **Basso rumore (σ=0.005-0.01)**: UNet dovrebbe ottenere ottimi risultati (PSNR ~28-32 dB)
  - L'architettura encoder-decoder e' progettata per catturare pattern gerarchici
  - Le skip connections preservano i dettagli fini persi nell'encoding
- **Medio rumore (σ=0.05)**: Buona qualita' (PSNR ~26-30 dB)
  - Il multi-noise training aiuta a generalizzare su diversi livelli di degrado
- **Alto rumore (σ=0.1)**: Qualita' ridotta (PSNR ~20-24 dB)
  - Il rumore elevato distrugge dettagli fini difficili da recuperare

### Vantaggi UNet
- **Inferenza veloce**: ~0.01 sec/img su GPU dopo training
- **End-to-end**: apprende direttamente la mappatura degradato → pulito
- **Skip connections**: preservano dettagli strutturali

### Limitazioni
- **Richiede training**: ~50 epoche, dipendenza dai dati di training
- **Generalizzazione**: performance dipendono dalla similarita' train/test
- **MSE loss**: tende a produrre immagini leggermente smooth (media statistica)
- **No physics**: non usa il modello di blur esplicito (a differenza di TV e DiffPIR)